# 04 - Model evaluation

**Goal.** Report what the models actually do on data they have never seen, and
be honest about how much of that performance is real.

Covered here:

1. Test-set metrics and confusion matrices.
2. Whether feature engineering answered the research question.
3. Which features matter, and whether they make physical sense.
4. A leakage check that the headline numbers do not capture.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/; the modules live in src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
print("project root:", PROJECT_ROOT)

## 4.1 Results produced by the pipeline

In [ ]:
from utils import METRICS_DIR, REPORTS_DIR, FEATURES_DIR
from data_loader import TARGET_SPECS

cv_results = pd.read_csv(METRICS_DIR / "model_results.csv")
test_results = pd.read_csv(METRICS_DIR / "test_results.csv")

display(test_results[[
    "display_name", "model", "feature_set", "accuracy",
    "f1_macro", "f1_weighted", "roc_auc_ovr_macro", "n_selected_features",
]].round(4))

**On ROC-AUC.** It is computed one-vs-rest and macro-averaged, and only when
that is technically appropriate — the model must expose probabilities and the
test set must contain every class, otherwise at least one curve is undefined.
Where those conditions fail the value is reported as `n/a` rather than silently
computed over a subset of classes.

## 4.2 Did feature engineering help?

In [ ]:
best = cv_results.groupby(["component", "feature_set"])["f1_macro"].max().unstack()
best = best[["baseline", "engineered", "selected"]]
best["gain_engineered"] = best["engineered"] - best["baseline"]
best["gain_selected"] = best["selected"] - best["baseline"]
display(best.round(4))

ax = best[["baseline", "engineered", "selected"]].plot.bar(figsize=(10, 4.5), rot=0)
ax.set_ylabel("best cross-validated macro F1")
ax.set_ylim(0.9, 1.005)
ax.set_title("Effect of feature engineering, by component")
ax.legend(title="feature set")

**Answer to the research question: yes, but selectively — and saying where it
does *not* help is part of the answer.**

- **Valve (+0.024)** and **accumulator (+0.022)** improve clearly. These are the
  targets the dataset's own documentation calls hard.
- **Cooler (-0.001)** does not. Cooling efficiency separates the classes almost
  linearly, so a per-sensor mean already solves it and there is nothing left to
  gain. Reporting this honestly is more useful than claiming a uniform win.
- **Pump (+0.004)** improves modestly.

A capstone that claimed feature engineering improved everything would be
overstating its own result.

## 4.3 Confusion matrices: which errors, not just how many

In [ ]:
import json
from utils import MODELS_DIR

for _, row in test_results.iterrows():
    component = row["component"]
    spec = TARGET_SPECS[component]
    meta = json.load(open(MODELS_DIR / component / "metadata.json"))
    path = REPORTS_DIR / f"classification_report_{component}.txt"
    print("=" * 78)
    print(path.read_text())

The targets are **ordered degradation levels**, so the *kind* of error matters
more than the count. Confusing "optimal" with "close to total failure" is an
operational disaster; confusing two adjacent levels is a minor error. Only the
confusion matrix distinguishes the two — which is why it is reported alongside
every aggregate metric.

A natural extension (noted in the README's future work) is to treat the targets
as genuinely **ordinal** so the loss function penalises distant confusions more.

## 4.4 Which features matter, and do they make physical sense?

In [ ]:
importance = pd.read_csv(METRICS_DIR / "feature_importance.csv")

for component in ["cooler", "valve", "pump", "accumulator"]:
    subset = importance[(importance.component == component) & (importance.kind == "impurity")]
    if subset.empty:
        subset = importance[importance.component == component]
    print(f"\n{TARGET_SPECS[component].display_name}")
    print(subset.head(8)[["rank", "feature_name", "importance"]].to_string(index=False))

**This is the sanity check that matters most.** A model can reach 100 % accuracy
for the wrong reasons; if the features it relies on are physically unrelated to
the fault, the result is an artefact. Here they line up:

| Component | Top features | Physics |
|---|---|---|
| Cooler | `CE_mean`, `CP_mean`, `PS6_mean` | the cooling-efficiency and cooling-power channels |
| Valve | `PS2_cv`, `PS2_band3_energy_ratio`, `PS2_max_abs_change` | switching lag appears as pressure variability and high-band spectral content |
| Pump | `FS1_q75`, `FS1_rms`, `EPS1_max` | internal leakage changes delivered flow and the power to produce it |
| Accumulator | `PS3_rollmean_max`, `PS1_band3_energy_ratio` | a discharged accumulator damps pressure transients less |

Note also that **impurity importance is biased** towards continuous,
high-cardinality features. The permutation importances — which measure the actual
drop in test macro F1 when a feature is shuffled — are the more trustworthy
ranking, and are saved in `outputs/figures/feature_importance/`.

## 4.5 The leakage check the headline numbers hide

In [ ]:
check = test_results[["display_name", "cv_f1_macro", "f1_macro", "block_cv_f1_macro"]].copy()
check["optimism"] = check["cv_f1_macro"] - check["block_cv_f1_macro"]
check.columns = ["component", "CV (random folds)", "test (random split)",
                 "temporal-block CV", "optimism"]
display(check.round(4))

Neighbouring cycles were recorded seconds apart under the same component
setting, so a random split can place near-duplicate cycles on both sides. The
block-grouped cross-validation holds out whole contiguous runs of 25 cycles, so
a cycle is never scored while its immediate neighbours sit in training.

**The accumulator loses roughly 7–8 points of macro F1** under that stricter
protocol. That gap is the honest estimate of how much of the apparent
performance comes from temporal proximity rather than from genuine
generalisation. Reporting only the random-split number would overstate what the
system would do on a freshly recorded cycle.

Recall from notebook 01 why a plain chronological split is not an option here:
the experiment was a designed sequence, and the final 20 % of cycles contain only
one cooler class.

## 4.6 PCA: is the engineered space separable at all?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from preprocessing import feature_columns
from utils import FEATURES_DIR

engineered = pd.read_csv(FEATURES_DIR / "engineered_features.csv")
columns = feature_columns(engineered)
X = StandardScaler().fit_transform(engineered[columns].to_numpy(float))
projected = PCA(n_components=2, random_state=0).fit_transform(X)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (key, spec) in zip(axes.ravel(), TARGET_SPECS.items()):
    labels = engineered[spec.column]
    for value in sorted(labels.unique()):
        mask = labels == value
        ax.scatter(projected[mask, 0], projected[mask, 1], s=8, alpha=0.6,
                   label=f"{value} - {spec.value_labels.get(int(value), '')}")
    ax.set_title(spec.display_name)
    ax.legend(fontsize=7)
fig.suptitle("PCA of the engineered feature space (scaled), coloured by each target")
fig.tight_layout()

**PCA is used for visualisation only, never as the final feature space.**
Principal components are linear mixtures of hundreds of features, so a model
built on them cannot tell a maintenance engineer *why* an alarm fired — which
defeats the purpose of a condition-monitoring system. As a picture, though, it
answers a useful question before any model runs: the cooler classes form obvious
separate clusters, while the accumulator classes overlap — exactly matching which
targets turn out to be easy and which are hard.